In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset
import torch.optim as optim
from torch.optim import lr_scheduler
import numpy as np
import torchvision
import torch.nn.functional as F
from torch.utils.data.sampler import SubsetRandomSampler
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from torchvision.transforms import Resize, ToTensor, Normalize
import matplotlib.pyplot as plt
#from imblearn.under_sampling import RandomUnderSampler

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score, \
    average_precision_score
from sklearn.model_selection import train_test_split
import time
import os
from pathlib import Path
from skimage import io
import copy
from torch import optim, cuda
import pandas as pd
import glob
from collections import Counter
# Useful for examining network
from functools import reduce
from operator import __add__
# from torchsummary import summary
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
from PIL import Image
from timeit import default_timer as timer
import matplotlib.pyplot as plt
import gc
from scipy import stats

In [2]:
import os
import sys
import pandas as pd
import torch
from torch.utils.data import DataLoader

crnn_dir = "/Users/tylerkei/DataScienceCapstoneECG/IntroECG/4-Models/CRNN-pytorch"
if crnn_dir not in sys.path:
    sys.path.append(crnn_dir)

from echonext_dataset import EchoNextWaveformDataset

project_data_dir = "/Users/tylerkei/DataScienceCapstoneECG/data/echonext_crnn"
meta_path = os.path.join(project_data_dir, "echonext_metadata_100k.csv")
train_waveform_path = os.path.join(project_data_dir, "EchoNext_train_waveforms.npy")
val_waveform_path = os.path.join(project_data_dir, "EchoNext_val_waveforms.npy")

meta = pd.read_csv(meta_path)

train_df_available = meta[meta["split"] == "train"].reset_index(drop=True)
eval_df_available = meta[meta["split"] == "val"].reset_index(drop=True)

y_train = train_df_available["shd_moderate_or_greater_flag"].astype("float32").to_numpy(copy=True)
y_test = eval_df_available["shd_moderate_or_greater_flag"].astype("float32").to_numpy(copy=True)

train_dataset = EchoNextWaveformDataset(train_waveform_path, y_train)
test_dataset = EchoNextWaveformDataset(val_waveform_path, y_test)

batch_size = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print("train samples:", len(train_dataset))
print("val samples:", len(test_dataset))
print("first train waveform shape:", train_dataset[0][0].shape)
print("first train label:", train_dataset[0][1].item())


ModuleNotFoundError: No module named 'echonext_dataset'

In [ ]:
try:
    del model
    gc.collect()
    torch.cuda.empty_cache()
except:
    pass

In [ ]:
from CRNN import CRNN
model = CRNN(hidR = 128 * 2, layerR = 1, hidC = 128 * 2)
# print(model)

In [ ]:
sum(p.numel() for p in model.parameters())

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# print(f'Train on: {device}')
# send model to device
model.to(device)

In [ ]:
def update_outputs(output_dict,
                   new_outputs,
                   outputs_to_update=['scores', 'predictions', 'labels']):
    for x in outputs_to_update:
        if (type(new_outputs[x]) is list):
            output_dict[x] += list(new_outputs[x])
        else:
            output_dict[x].append(new_outputs[x])
    return output_dict

def accuracyFromLogits(output, target):
    scores = torch.sigmoid(output)
    pred = torch.round(scores)
    correct_tensor = pred.eq(target.data.view_as(pred))
    # Need to convert correct tensor from int to float to average
    accuracy = torch.mean(correct_tensor.type(torch.FloatTensor))
    return accuracy, scores, pred

In [ ]:
def train(model,
          device,
          criterion,
          optimizer,
          train_loader,
          valid_loader,
          save_file_name,
          max_epochs_stop=5,
          n_epochs=40,
          print_every=1):
    """Train a PyTorch Model

    Params
    --------
        model (PyTorch model): cnn to train
        criterion (PyTorch loss): objective to minimize
        optimizer (PyTorch optimizier): optimizer to compute gradients of model parameters
        train_loader (PyTorch dataloader): training dataloader to iterate through
        valid_loader (PyTorch dataloader): validation dataloader used for early stopping
        save_file_name (str ending in '.pt'): file path to save the model state dict
        max_epochs_stop (int): maximum number of epochs with no improvement in validation loss for early stopping
        n_epochs (int): maximum number of training epochs
        print_every (int): frequency of epochs to print training stats

    Returns
    --------
        model (PyTorch model): trained cnn with best weights
        history (DataFrame): history of train and validation loss and accuracy
    """

    # Early stopping intialization
    epochs_no_improve = 0
    valid_loss_min = np.inf

    # Create empty history
    history = []

    # Number of epochs already trained (if using loaded in model weights)
    try:
        print(f'Model has been trained for: {model.epochs} epochs.\n')
    except:
        model.epochs = 0
        print(f'Starting Training from Scratch on \n')

    overall_start = timer()

    # Main loop
    for epoch in range(n_epochs):
        model.train()
        # keep track of training and validation loss each epoch
        train_step_counter = 0
        valid_step_counter = 0

        train_loss = 0.0
        valid_loss = 0.0

        train_acc = 0
        valid_acc = 0
        running_corrects = 0

        # Set to training
        # model.train()
        start = timer()

                # Don't need to keep track of gradients
        train_output_dict = {
            "scores": [],
            "predictions": [],
            "labels": []
        }

        # Training loop
        for ii, (input, target) in enumerate(train_loader):

            # Increment counter
            train_step_counter += 1

            # Set inputs dtype. Check input.dtype and target.dtype
            # NOTE: This match the type of input and model weights
            input, target = input.to(torch.float32), target.to(torch.float32)

            # Send inputs to device
            input, target = input.to(device), target.to(device)

            # Clear gradients
            optimizer.zero_grad()

            # Calculate output
            output = model(input)

            # Loss and backpropagation of gradients
            # WARNING: Be careful about .squeeze(). In this case, since the output is
            # [BATCHSIZE, 1] the scenario is easy!
            matched_output = output.squeeze()
            assert matched_output.shape == target.shape
            loss = criterion(matched_output, target) # save a copy, plot the

            # Backward computation
            loss.backward()

            # Update the parameters
            optimizer.step()

            # Track train loss by multiplying average loss by number of examples in batch
            train_loss += loss.item()

            # Calculate accuracy by finding max log probability
            accuracy, scores, pred = accuracyFromLogits(output, target)
            train_acc += accuracy.item()

            train_output_dict = update_outputs(train_output_dict, new_outputs={
                    "predictions": pred.detach().cpu().numpy().squeeze().tolist(),
                    "scores": scores.detach().cpu().numpy().squeeze().tolist(),
                    "labels": target.detach().cpu().numpy().squeeze().tolist()
                })

            # Track training progress
            print(
                f'Epoch: {epoch}\t{100 * (ii + 1) / len(train_loader):.2f}% complete. {timer() - start:.2f} seconds elapsed in epoch.',
                end='\r')

        # Calculate accuracy and loss over the batch
        train_acc = train_acc / train_step_counter
        train_loss = train_loss / train_step_counter


        # After training loops ends, start validation
        model.epochs += 1

        # Don't need to keep track of gradients
        output_dict = {
            "scores": [],
            "predictions": [],
            "labels": []
        }
        with torch.no_grad():
            # Set to evaluation mode
            model.eval()

            # Validation loop
            for input, target in valid_loader:

                # Increment counter
                valid_step_counter += 1

                # Set inputs dtype. Check input.dtype and target.dtype
                # NOTE: This match the type of input and model weights
                input, target = input.to(torch.float32), target.to(torch.float32)

                # Send inputs to device
                input, target = input.to(device), target.to(device)

                # Calculate output
                output = model(input)

                # Loss and backpropagation of gradients
                loss = criterion(output, target) # save copy

                # Track train loss by multiplying average loss by number of examples in batch
                valid_loss += loss.item()

                # Calculate accuracy by finding max log probability
                accuracy, scores, pred = accuracyFromLogits(output, target)
                valid_acc += accuracy.item()

                output_dict = update_outputs(output_dict, new_outputs={
                    "predictions": pred.detach().cpu().numpy().squeeze().tolist(),
                    "scores": scores.detach().cpu().numpy().squeeze().tolist(),
                    "labels": target.detach().cpu().numpy().squeeze().tolist()
                })

            # Calculate accuracy and loss over the batch
            valid_acc = valid_acc / valid_step_counter
            valid_loss = valid_loss / valid_step_counter

            # Calculate average accuracy
            roc_auc_curve = roc_auc_score(output_dict["labels"], output_dict["scores"])
            train_roc_auc_curve = roc_auc_score(train_output_dict["labels"], train_output_dict["scores"])
            history.append([train_loss, valid_loss, train_acc, valid_acc, train_roc_auc_curve, roc_auc_curve])

        # Print training and validation results
        if (epoch + 1) % print_every == 0:
            # print("epoch_acc",epoch_acc)
            print("\n---------------------------------------------------------------------------------------------"
                  f'Epoch {epoch} Results'
                  "---------------------------------------------------------------------------------------------\n")
            print(
                f'Training Loss: {train_loss:.4f} \t Validation Loss: {valid_loss:.4f}'
            )
            print(
                f'Training Accuracy: {100 * train_acc:.2f}%\t Validation Accuracy: {100 * valid_acc:.2f}%'
            )

            precision, recall, f1_score, _ = precision_recall_fscore_support(output_dict["labels"],
                                                                             output_dict["predictions"])
            print(f'Training roc_auc_score:{train_roc_auc_curve}')
            print(f'roc_auc_score :{roc_auc_curve}')

            for j in range(len(precision)):
                print(f'\nClass {j} Precision :{precision[j] * 100:.2f} ')
                print(f'Class {j} Recall :{recall[j] * 100:.2f} ')
                print(f'Class {j} f1_score :{f1_score[j] * 100:.2f} ')

        # Save the model if validation loss decreases
        if valid_loss < valid_loss_min:
            # Save model
#             torch.save(model.state_dict(), save_file_name)
            # Track improvement
            epochs_no_improve = 0
            valid_loss_min = valid_loss
            valid_best_acc = valid_acc
            best_epoch = epoch

        # Otherwise increment count of epochs with no improvement
        else:
            epochs_no_improve += 1


    # # Attach the optimizer
    # model.optimizer = optimizer
    # Record overall time and print out stats
    total_time = timer() - overall_start
    print(
        f'\nBest epoch: {best_epoch} with loss: {valid_loss_min:.2f} and acc: {100 * valid_acc:.2f}%'
    )
    print(
        f'{total_time:.2f} total seconds elapsed. {total_time / (epoch):.2f} seconds per epoch.'
    )
    # eval_metrics_dict, b = (get_metrics(eval_outputs_dict))
    # for idx, class_idx in enumerate(eval_metrics_dict):
    #     for metric_name in eval_metrics_dict[class_idx]:
    #         print("Class ", class_idx, "Evaluation", metric_name, ":", eval_metrics_dict[class_idx][metric_name])

    print("\n\n----------------\n\n")
    # Format history
    history = pd.DataFrame(
        history,
        columns=['train_loss', 'valid_loss', 'train_acc', 'valid_acc', 'train_roc', 'roc'])
    print("history df shape: ", history.shape)
    return model, history

In [ ]:
criterion = nn.BCEWithLogitsLoss()
criterion.to(device)

optimizer_ft = optim.Adam(model.parameters(), weight_decay=0.001)

NUM_EPOCHS = 2
save_file_name = 'echonext_crnn.pt'


In [ ]:
model, history = train(
    model,
    device,
    criterion,
    optimizer_ft,
    train_loader,
    test_loader,
    save_file_name=save_file_name,
    max_epochs_stop=2,
    n_epochs=NUM_EPOCHS,
    print_every=1
)

In [ ]:
def plot(history):

  # The history object contains results on the training and test
  # sets for each epoch
  acc = history['train_acc']
  val_acc = history['valid_acc']
  loss = history['train_loss']
  val_loss = history['valid_loss']

  # Get the number of epochs
  epochs = range(len(acc))

  plt.style.use('seaborn')

  plt.title('Training and validation accuracy')
  plt.plot(epochs, acc, color='blue', label='Train')
  plt.plot(epochs, val_acc, color='orange', label='Val')
  plt.xlabel('Epoch')
  plt.ylabel('Accuracy')
  plt.legend()

  _ = plt.figure()
  plt.title('Training and validation loss')
  plt.plot(epochs, loss, color='blue', label='Train')
  plt.plot(epochs, val_loss, color='orange', label='Val')
  plt.xlabel('Epoch')
  plt.ylabel('Loss')
  plt.legend()

  _ = plt.figure()
  plt.title('Training and validation ROC AUC')
  plt.plot(epochs, history['train_roc'], color='blue', label='Train')
  plt.plot(epochs, history['roc'], color='orange', label='Val')
  plt.xlabel('Epoch')
  plt.ylabel('Loss')
  plt.legend()

In [ ]:
plot(history)   # default settings

In [ ]:
# hidR = 128 * 2, layerR = 1, hidC = 128 * 2, dropout = 0.5, l2 = 0.001
# ROC: 0.84904，0.85640, 0.85133
plot(history)
print("Best ROC AUC",history['roc'].max())
print("Best Acc", history['valid_acc'].max())
print("----Best Epoch:")
print(history.loc[history['roc'].idxmax(), :])
print("----")

In [ ]:
# hidR = 128 * 2, layerR = 1, hidC = 128 * 2, dropout = 0.5
# ROC: 0.842, 0.8512, 0.85453, 0.86790
plot(history)
print("Best ROC AUC",history['roc'].max())
print("Best Acc", history['valid_acc'].max())
print("----Best Epoch:")
print(history.loc[history['valid_acc'].idxmax(), :])
print("----")

In [ ]:
plt.style.use('seaborn')

In [ ]:
# hidR = 128 * 2, layerR = 1, hidC = 128 * 2
plot(history)
print("Best ROC AUC",history['roc'].max())
print("Best Acc", history['valid_acc'].max())

In [ ]:
# hidR = 128 * 2, layerR = 1, hidC = 128 * 4
plot(history)
print("Best ROC AUC",history['roc'].max())
print("Best Acc", history['valid_acc'].max())

In [ ]:
# LSTM
# hidR = 128 * 2, layerR = 1, hidC = 128 * 2, dropout = 0.5
# 0.8030602355747727
# Best Acc 0.7956688602765402

plot(history)
print("Best ROC AUC",history['roc'].max())
print("Best Acc", history['valid_acc'].max())
print("----Best Epoch:")
print(history.loc[history['valid_acc'].idxmax(), :])
print("----")